# 16.7 精度能降低嗎？


同一個權重若用四bytes保存，與用兩bytes保存，前者通常能分辨更細的差異；後者省空間，也可能讓支援它的硬體更快。但兩種「兩bytes浮點數」不一定能表示相同範圍。先回顧[數值尺度](https://birdhackor.github.io/tiny-perceptron-vlm/14.2.html)：後續運算不只在乎形狀，還會受數字大小影響。本節要觀察較短的浮點表示如何改變一次矩陣乘法。

FP32是32位元浮點數，一個數字四bytes；FP16與BF16都是16位元、兩bytes。浮點表示可粗略想成「有效數字×某個次方」：存較多有效數字的位元，附近刻度更細；存較多次方的位元，可表示的極大與極小數範圍更寬。FP16通常比BF16有更細的相對精度，但最大有限值約65504；BF16保留較寬的次方範圍，最大值約3.39×10³⁸，接近FP32。

用固定輸入做CPU的BF16小實驗。第一個矩陣的數字刻意接近整數，BF16可能把1.001等值捨入到1，讓輸出有可追蹤的差異。`autocast`是PyTorch依運算選擇較低精度的介面；這裡只在包住的矩陣乘法使用，不手動改原輸入的儲存型別。

`a @ b`是矩陣乘法：取a的一列與b的一欄，對應項相乘再相加，得到輸出的同一列、同一欄。例如左上格是 `1.001×0.5 + 2.002×1.5 = 3.5035`；右上格改取b的第二欄，便是 `1.001×1 + 2.002×(-1) = -1.001`。第二列用a的3.003與4.004照同一規則計算。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

a = torch.tensor([[1.001, 2.002], [3.003, 4.004]])
b = torch.tensor([[0.5, 1.0], [1.5, -1.0]])
full = a @ b
with torch.autocast("cpu", dtype=torch.bfloat16):
    low = a @ b
print("原輸入型別", a.dtype, "低精度輸出", low.dtype)
print("FP32", full.round(decimals=4).tolist())
print("BF16", low.float().tolist())
print("最大差異", round((full - low.float()).abs().max().item(), 4))

第一行應看到原輸入仍torch.float32，輸出是torch.bfloat16。FP32結果約 `[[3.5035,-1.001],[7.5075,-1.001]]`，BF16約 `[[3.5,-1],[7.5,-1]]`，最大差異約0.0075。`low.float()`把結果轉回FP32，只為方便比較，不能恢復先前已被捨入丟掉的細節。相減得到四格各自的差異，`.abs()`取絕對值、`.max()`選最大一格、`.item()`取出普通Python數字，最後四捨五入到四位小數。這個差異不是一層權重學壞了，而是表示與運算精度改變。

混合精度訓練通常讓合適的矩陣運算用低精度，某些誤差計算與累加仍用FP32。[梯度](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)是損失對參數的變化率，由反向傳播沿計算關係求出，用來決定更新方向。FP16計算中的微小梯度可能太小而無法表示，變成零，這叫下溢。GradScaler是管理這個尺度的工具：先把損失乘上倍率，例如1024，再反向傳播算出同樣放大1024倍的梯度，更新參數前再除以同一倍率，縮回原尺度。放大是為了減少計算途中小量變成零的情形，不是把學習率任意放大1024倍。這是額外的數值管理，不能只改dtype就假設訓練安全。BF16範圍較寬，也仍會有捨入誤差；每個硬體支援的運算不同，應核對結果與實際耗時。

我們另以[T.4](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)的直接SFT起點做了L4真實訓練：寬度64、兩層、原本的一個注意力頭、最多128位置，45／5／10題家族切分。它沒有接在上面的四頭efficiency模型之後；三支各重新載入同一個單頭FP32模型，seed 42、每批16題、AdamW學習率0.003，均完成200次嘗試、計22,493個有效訓練目標。

| 訓練方式 | 成功更新／跳過 | 驗證／最後檢查代價 | 完整答案匹配：驗證／最後檢查 | 毫秒／訓練步 | 毫秒／向前 |
| --- | --- | --- | --- | ---: | ---: |
| FP32 | 200／0 | 0.47748／0.36272 | 2/5／7/10 | 11.673 | 2.444 |
| BF16 autocast | 200／0 | 0.83820／0.32709 | 1/5／6/10 | 14.414 | 2.835 |
| FP16 autocast＋GradScaler | 200／0 | 1.17649／0.71923 | 1/5／4/10 | 14.280 | 2.740 |

三支的模型參數始終是FP32，141,568個權重數字共566,272 bytes，Adam的浮點歷史狀態也保留FP32；並沒有把整個模型靜態轉成兩bytes。實際向前輸出的dtype分別是FP32、BF16與FP16，訓練loss皆有限，最後參數與核對logit也有限，FP16使用scaler且本輪沒有跳步。有限值核對是排除兩類特殊值：`inf`表示無窮大，例如數值超出可表示範圍；`NaN`是「不是有效數字」的標記，例如PyTorch浮點tensor中0除以0的結果。它們不同於普通捨入誤差，也不同於下溢得到的有限值0。依[PyTorch 2.14 AMP文件的autocast與Gradient Scaling段落](https://docs.pytorch.org/docs/2.14/amp.html)，scaler遇到inf／NaN梯度會跳過optimizer更新，所以一般情況下「完成200次嘗試」仍須另核對成功／跳過，不能預設等於200次更新。

表中代價按助手回答與EOS計，驗證36、最後檢查69個有效目標；完整答案則以5／10題原始ID匹配，最多生成32個新ID，三支這兩側都正常EOS結束。BF16最後檢查代價比FP32低，答對數卻少一題；FP16不只代價較高，第一筆理想答案`circle`還生成`circlow`，同一題FP32生成`circle`。有限值與正常結束並不保證答案正確。

時間沿用[16.1](https://birdhackor.github.io/tiny-perceptron-vlm/16.1.html)的同步範圍：訓練略過前三步取中位數，含梯度有限值核對與更新；向前則取五題驗證的同一補齊批次，暖機三次、量九次。在這個很小的模型上，兩種AMP都較慢。我們沒有另測大型矩陣，也沒有逐步增加每批資料、檢查每秒處理量何時不再增加；因此不知道充分忙碌的硬體會有什麼結果，不能由這輪直接外推。

| 設定 | 支線開始前已配置MiB | 該支線訓練峰值MiB | 峰值減起點MiB |
| --- | ---: | ---: | ---: |
| FP32 | 33.086 | 79.261 | 46.175 |
| BF16 | 65.374 | 75.882 | 10.509 |
| FP16 | 65.232 | 75.743 | 10.510 |

各欄由原始bytes獨立四捨五入到三位小數。各支線都重新reset配置器峰值，但逐支執行的駐留／工作區起點不同，差約32 MiB，不能把新增量差全說成計算時保存的中間特徵與輸出省了多少，更不能宣稱權重減半。數字也不含reserved未使用空間、driver及框架之外的配置。[precision完整實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/precision.json)保留排程、scaler history、實際dtype、留出生成與各支線數值；這輪單種子短續訓只能說明本次取捨。

練習先把a的建立行改成 `a = torch.tensor([[1.001, 2.002], [3.003, 4.004]]) * 10`，從頭執行。先預測最大絕對差異大致放大，而非保持0.0075，再看實際捨入結果。第二次則改成 `a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])`：數值都是整數，但用小數點寫法保留FP32儲存型別，讓它仍能與浮點的b相乘；預測本例兩路更接近，再從頭執行核對。CPU數字核對能說明精度取捨，不能證明GPU加速或整個模型品質保持不變。
